# GDS Import, Generation, Export, And EMX Port Mapping

This public notebook shows the portable GDS side of an `emx-dbs` study. It covers three common tasks:

- Import and inspect an existing GDS file.
- Generate a custom parametric GDS seed, using the dual-core VCO tank as the example.
- Export a configured square-pixel candidate GDS and understand the EMX port/proc/command mapping.

All generated files are written under `local/notebooks/gds_import_generation_export_emx_ports/`, which is ignored by git. The notebook uses the fake EMX backend by default so it can run on any machine without Cadence/EMX installed.


In [ ]:
from __future__ import annotations

import json
import shlex
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import yaml
from IPython.display import Image, display

# Find the source checkout even when Jupyter starts in notebooks/.
repo_root = Path.cwd().resolve()
for candidate in (repo_root, *repo_root.parents):
    if (candidate / 'pyproject.toml').exists() and (candidate / 'emx_dbs').exists():
        repo_root = candidate
        break
else:
    raise RuntimeError('Could not find the emx-dbs repository root')

if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

out_dir = repo_root / 'local' / 'notebooks' / 'gds_import_generation_export_emx_ports'
out_dir.mkdir(parents=True, exist_ok=True)

print('Repository root:', repo_root)
print('Notebook output:', out_dir)


In [ ]:
from emx_dbs.config import load_config, prepare_run_dir
from emx_dbs.dbs import evaluate_masks, eval_one
from emx_dbs.emx_runner import frequency_sweep_ghz, validate_emx_environment
from emx_dbs.gds_io import inspect_gds, inspect_raw_gds, write_candidate_gds
from emx_dbs.legality import check_legality
from emx_dbs.mutate import apply_flips
from emx_dbs.rasterize import rasterize_config
from emx_dbs.reporting import generate_report, write_gds_preview, write_layout_preview
from emx_dbs.tank_generator import (
    DualCoreVcoTankGeometry,
    generate_dual_core_vco_tank_gds,
    write_dual_core_vco_tank_config,
)


## 1. Import And Inspect An Existing GDS

Start by pointing `INPUT_GDS` and `INPUT_TOP_CELL` at any layout you want to inspect. The default uses the small public example seed in this repository. `inspect_raw_gds` does not need an optimizer config; it reports cells, layers, polygon counts, label counts, bounding boxes, and area by raw GDS layer/datatype.


In [ ]:
INPUT_GDS = repo_root / 'examples' / 'generic_nport' / 'seed.gds'
INPUT_TOP_CELL = 'TOP'

raw_input_info = inspect_raw_gds(INPUT_GDS, top_cell=INPUT_TOP_CELL)
print(json.dumps(raw_input_info, indent=2, sort_keys=True))

input_preview = out_dir / 'input_gds_preview.png'
write_gds_preview(INPUT_GDS, input_preview, top_cell=INPUT_TOP_CELL, show_legend=True, show_title=True)
display(Image(filename=str(input_preview)))


## 2. Generate A Custom Dual-Core VCO Tank GDS

The generator below creates a square-pixel dual-core VCO tank seed. The physical layer defaults are N16-oriented examples: M9 on GDS `39/60`, M8 on `38/40`, V8 on `58/60`, and an optional lower-metal guard ring on `31/0`. For another process, change these GDS layer/datatype pairs to match the EMX process file used by that environment.

The port labels follow a differential tank convention:

- `PP`, `PN`: primary positive/negative feed labels.
- `SP`, `SN`: secondary positive/negative feed labels.
- `GS`, `GN`: optional guard or ground reference labels when guard ports are enabled.

The names are ordinary GDS text labels. Keep the names in the GDS, YAML `ports` list, objective parameters, and EMX command aligned.


In [ ]:
geom = DualCoreVcoTankGeometry(
    top_cell='public_dual_core_vco_tank_ring33',
    core_width_um=330.0,
    core_height_um=330.0,
    pitch_um=10.0,
    m9_ring_width_um=20.0,
    center_gap_um=10.0,
    feed_spacing_um=20.0,
    feed_width_um=5.0,
    feed_length_um=26.0,
    m8_trace_width_um=10.0,
    pixel_style='square',
    emit_v8=True,
    include_guard_ring=True,
    guard_layer=(31, 0),
    guard_offset_um=26.0,
    guard_width_um=10.0,
    guard_feed_overlap_um=5.0,
    include_guard_ports=True,
)

seed_gds = out_dir / 'public_dual_core_vco_tank_ring33.gds'
config_path = out_dir / 'public_dual_core_vco_tank_ring33.local.yaml'
raw_seed_gds = generate_dual_core_vco_tank_gds(seed_gds, geom)
write_dual_core_vco_tank_config(
    config_path,
    raw_seed_gds,
    geom,
    run_id='public_dual_core_vco_tank_ring33_demo',
    output_root=out_dir / 'runs',
)

print('Generated seed GDS:', seed_gds)
print('Generated local config:', config_path)
print(json.dumps(inspect_raw_gds(seed_gds, top_cell=geom.top_cell), indent=2, sort_keys=True))

seed_preview = out_dir / 'generated_tank_raw_preview.png'
write_gds_preview(seed_gds, seed_preview, top_cell=geom.top_cell, show_legend=True, show_title=True)
display(Image(filename=str(seed_preview)))


## 3. Read The Generated `emx-dbs` Config

The YAML config is the contract between GDS, pixelization, EMX, and the objective. Logical layer names such as `m9`, `m8`, and `v8` are local handles used by `emx-dbs`; EMX itself only sees the GDS layer/datatype numbers and the process file. The process file must define those GDS layers as the intended stack metals/vias.


In [ ]:
cfg = load_config(config_path)
print(json.dumps(validate_emx_environment(cfg), indent=2, sort_keys=True, default=str))
print(json.dumps(inspect_gds(cfg), indent=2, sort_keys=True, default=str))

layer_table = pd.DataFrame(
    [
        {'logical_layer': name, 'gds_layer': spec[0], 'gds_datatype': spec[1]}
        for name, spec in cfg.layers.items()
    ]
)
port_table = pd.DataFrame(
    [
        {
            'touchstone_index': idx,
            'name': port.name,
            'logical_layer': port.layer,
            'xy_um': tuple(port.xy_um),
            'edge': port.edge,
            'width_um': port.width_um,
        }
        for idx, port in enumerate(cfg.ports, start=1)
    ]
)

print('Layer mapping used by emx-dbs and expected by the EMX proc file:')
display(layer_table)
print('Port order used for result.sNp and objective port numbers:')
display(port_table)


## 4. Rasterize The Seed And Export A Candidate GDS

Rasterization turns the configured GDS layers inside mutable windows into boolean masks. Exporting writes a new GDS from those masks. This is the same path used before each EMX evaluation.

Corner-overlap bridge support is controlled by the config DRC fields `allow_same_layer_diagonal_contact` and `corner_overlap_bridge`. When enabled, diagonal-only same-layer pixel contacts are treated as connected and exported with small bridge polygons.


In [ ]:
maskset = rasterize_config(cfg)
run_dir = prepare_run_dir(cfg, config_path)
preview_path = out_dir / 'rasterized_layout_preview.png'
exported_gds = out_dir / 'exported_square_candidate.gds'

write_layout_preview(maskset, preview_path, cfg, annotate_geometry=True, show_legend=True, show_title=True)
write_candidate_gds(maskset, cfg, exported_gds)

print('Rasterized preview:', preview_path)
print('Exported candidate GDS:', exported_gds)
print(json.dumps(inspect_raw_gds(exported_gds, top_cell=cfg.layout.top_cell), indent=2, sort_keys=True))
display(Image(filename=str(preview_path)))


## 5. EMX Port And Process-File Mapping

`emx-dbs` builds EMX internal-port flags from the YAML `ports` list. For each port with `width_um`, the generated EMX script contains an option like `--internal=PP,5`. The labels must exist in the exported GDS on the configured port layer.

The EMX process file is passed as the final process argument after the GDS path and top-cell name. It is responsible for mapping raw GDS layer/datatype pairs to the real metal/via stack. For example, if this notebook says logical `m9` is GDS `39/60`, then the process file used by EMX must interpret `39/60` as the intended top metal.

Touchstone numbering follows the YAML port order shown above. With the generated tank defaults, the first four ports are `PP`, `PN`, `SP`, `SN`, so a differential objective usually uses primary ports `[1, 2]` and secondary ports `[3, 4]`. Guard ports, when present, come after the signal ports.


In [ ]:
def emx_command_preview(cfg, gds_path, results_dir, *, proc_file='<absolute-path-to-process-file.proc>', key='<optional-key-or-omit>'):
    nports = max(1, len(cfg.ports))
    touchstone = Path(results_dir) / f'result.s{nports}p'
    internal_args = [f'--internal={port.name},{port.width_um:g}' for port in cfg.ports if port.width_um is not None]
    key_args = [] if key in {None, '', '<optional-key-or-omit>'} else [f'--key={key}']
    freqs_hz = [freq_ghz * 1e9 for freq_ghz in frequency_sweep_ghz(cfg)]
    parts = [
        cfg.emx.executable,
        '--touchstone',
        f'--s-file={touchstone}',
        '--include-command-line',
        '--verbose=2',
        *key_args,
        *internal_args,
        *cfg.emx.extra_args,
        str(gds_path),
        cfg.layout.top_cell,
        proc_file,
        *[f'{freq_hz:.12g}' for freq_hz in freqs_hz],
    ]
    return ' '.join(shlex.quote(str(part)) for part in parts)

print(emx_command_preview(cfg, exported_gds, out_dir / 'emx_preview_results'))


## 6. Objective Function And FoM For A Dual-Core VCO DBS Trial

For the dual-core VCO tank, a useful first DBS FoM is differential-mode Q at a target design frequency. This notebook uses `TARGET_FREQ_GHZ = 11.0`.

The objective reads the EMX Touchstone result, converts the S-parameter matrix to an impedance matrix, and computes each differential pair as:

```text
Zdd(PP, PN) = Z_PP,PP - Z_PP,PN - Z_PN,PP + Z_PN,PN
Q_primary   = abs(Im(Zdd(PP, PN))) / Re(Zdd(PP, PN))

Zdd(SP, SN) = Z_SP,SP - Z_SP,SN - Z_SN,SP + Z_SN,SN
Q_secondary = abs(Im(Zdd(SP, SN))) / Re(Zdd(SP, SN))
```

The FoM is maximized. By default this section uses `min(Q_primary, Q_secondary)` so the optimizer cannot improve one side while neglecting the other. Port numbers come from the YAML order: `PP=1`, `PN=2`, `SP=3`, `SN=4` for the generated tank.


In [ ]:
TARGET_FREQ_GHZ = 11.0
Q_AGGREGATE = 'min'
Q_BALANCE_WEIGHT = 0.0
Q_MIN_REAL_OHM = 1.0e-3

objective_module = out_dir / 'vco_tank_q_objective.py'
objective_module.write_text(
    '''from pathlib import Path

import numpy as np

from emx_dbs.schemas import ObjectiveResult
from emx_dbs.touchstone import read_touchstone


def _nearest_frequency_index(freq_ghz, target_ghz):
    return int(np.argmin(np.abs(np.asarray(freq_ghz, dtype=float) - float(target_ghz))))


def _s_to_z(s_matrix, z0):
    s_matrix = np.asarray(s_matrix, dtype=complex)
    eye = np.eye(s_matrix.shape[0], dtype=complex)
    return float(z0) * (eye + s_matrix) @ np.linalg.inv(eye - s_matrix)


def _zdd(z_matrix, ports):
    a = int(ports[0]) - 1
    b = int(ports[1]) - 1
    return z_matrix[a, a] - z_matrix[a, b] - z_matrix[b, a] + z_matrix[b, b]


def _q_from_zdd(zdd, min_real_ohm):
    real = float(np.real(zdd))
    imag = float(np.imag(zdd))
    if not np.isfinite(real) or not np.isfinite(imag) or real <= 0.0:
        return float('nan')
    return abs(imag) / max(real, float(min_real_ohm))


def _aggregate(q_primary, q_secondary, params):
    mode = str(params.get('aggregate', 'min'))
    balance_weight = float(params.get('balance_weight', 0.0))
    if mode == 'min':
        base = min(q_primary, q_secondary)
    elif mode == 'mean':
        base = 0.5 * (q_primary + q_secondary)
    elif mode == 'mean_minus_balance':
        base = 0.5 * (q_primary + q_secondary)
        if balance_weight == 0.0:
            balance_weight = 0.5
    else:
        raise ValueError(f'Unsupported aggregate mode: {mode}')
    return float(base - balance_weight * abs(q_primary - q_secondary))


def dual_core_vco_differential_q(touchstone_path, metadata, params):
    sp = read_touchstone(Path(touchstone_path))
    primary_ports = params.get('primary_ports', [1, 2])
    secondary_ports = params.get('secondary_ports', [3, 4])
    required_nports = max(max(primary_ports), max(secondary_ports))
    if sp.nports < required_nports:
        return ObjectiveResult(fom=-1e30, loss=1e30, valid=False, reason='requires_configured_differential_ports')

    target_ghz = float(params.get('target_freq_ghz', 11.0))
    freq_ghz = sp.frequency_hz / 1e9
    idx = _nearest_frequency_index(freq_ghz, target_ghz)
    z = _s_to_z(sp.s[idx], sp.z0)

    zdd_primary = _zdd(z, primary_ports)
    zdd_secondary = _zdd(z, secondary_ports)
    q_primary = _q_from_zdd(zdd_primary, params.get('min_real_ohm', 1.0e-3))
    q_secondary = _q_from_zdd(zdd_secondary, params.get('min_real_ohm', 1.0e-3))
    valid = bool(np.isfinite(q_primary) and np.isfinite(q_secondary))
    metrics = {
        'target_freq_ghz': target_ghz,
        'actual_freq_ghz': float(freq_ghz[idx]),
        'q_primary': float(q_primary),
        'q_secondary': float(q_secondary),
        'q_balance_abs': float(abs(q_primary - q_secondary)) if valid else float('nan'),
        'zdd_primary_real_ohm': float(np.real(zdd_primary)),
        'zdd_primary_imag_ohm': float(np.imag(zdd_primary)),
        'zdd_secondary_real_ohm': float(np.real(zdd_secondary)),
        'zdd_secondary_imag_ohm': float(np.imag(zdd_secondary)),
        'aggregate': str(params.get('aggregate', 'min')),
    }
    if not valid:
        return ObjectiveResult(fom=-1e30, loss=1e30, valid=False, reason='invalid_differential_q', metrics=metrics)

    fom = _aggregate(q_primary, q_secondary, params)
    return ObjectiveResult(fom=fom, loss=-fom, valid=True, reason=None, metrics=metrics)
''',
    encoding='utf-8',
)
if str(out_dir) not in sys.path:
    sys.path.insert(0, str(out_dir))

q_config = yaml.safe_load(config_path.read_text())
q_config['run']['run_id'] = 'public_dual_core_vco_tank_ring33_q_symmetric_demo'
q_config['emx']['backend'] = 'fake'
q_config['emx']['freq_start_ghz'] = 9
q_config['emx']['freq_stop_ghz'] = 13
q_config['emx']['freq_step_ghz'] = 1
q_config['objective'] = {
    'plugin': 'vco_tank_q_objective:dual_core_vco_differential_q',
    'params': {
        'primary_ports': [1, 2],
        'secondary_ports': [3, 4],
        'target_freq_ghz': TARGET_FREQ_GHZ,
        'aggregate': Q_AGGREGATE,
        'balance_weight': Q_BALANCE_WEIGHT,
        'min_real_ohm': Q_MIN_REAL_OHM,
    },
}
q_config['dbs'].update(
    {
        'max_evaluations': 5,
        'max_rejections_in_a_row': 5,
        'metal_flip_count_weights': [1.0],
        'metal_flip_count_values': [8],
        'random_seed': 11033,
    }
)

q_config_path = out_dir / 'public_dual_core_vco_tank_ring33_q.local.yaml'
q_config_path.write_text(yaml.safe_dump(q_config, sort_keys=False), encoding='utf-8')

# From this point forward, use the Q-objective config for run examples.
config_path = q_config_path
cfg = load_config(config_path)
run_dir = prepare_run_dir(cfg, config_path)

print('Objective module:', objective_module)
print('Active Q-objective config:', config_path)
print('Active run dir:', run_dir)
print(json.dumps({'objective': q_config['objective'], 'emx': q_config['emx']}, indent=2, sort_keys=True))


## 7. Y-Axis Symmetric M9-Only DBS Trial

For this tank family, DBS proposals should preserve y-axis symmetry. In layout coordinates here, that is mirror symmetry about the vertical centerline at `x = 165 um`. The short trial below samples M9-only flip groups where every selected pixel is paired with its mirrored pixel before evaluation.

The trial also uses the full `emx-dbs` legality check before EMX. That means a proposal is rejected if it shorts configured forbidden ports or if an active V8 via is no longer enclosed by active M8 and M9 at the via center. The latter prevents physically invalid candidates where a via remains after one of its adjacent metal pads was flipped off.

This is deliberately a small notebook-level trial. It demonstrates the constraint and FoM plumbing using the fake backend. For real EMX, keep the same symmetry sampler but switch the config to `emx.backend: real` and provide local EMX setup paths.


In [ ]:
def y_axis_mirror_index(maskset, layer, row, col):
    grid = maskset.grids[layer]
    x_um, y_um = grid.index_center(int(row), int(col))
    mirror_x_um = grid.xmin + grid.xmax - x_um
    return grid.xy_to_index(mirror_x_um, y_um)


def assert_y_axis_symmetric(maskset, layer='m9'):
    mask = maskset.masks[layer]
    for row in range(mask.shape[0]):
        for col in range(mask.shape[1]):
            mirror = y_axis_mirror_index(maskset, layer, row, col)
            if mirror is None or bool(mask[row, col]) != bool(mask[mirror]):
                raise AssertionError((layer, row, col, mirror, 'not_y_axis_symmetric'))


def symmetric_m9_flip_groups(maskset):
    mutable = maskset.mutable_masks['m9']
    rows, cols = np.nonzero(mutable)
    groups = []
    seen = set()
    for row, col in zip(rows.tolist(), cols.tolist()):
        mirror = y_axis_mirror_index(maskset, 'm9', int(row), int(col))
        if mirror is None:
            continue
        mirror_row, mirror_col = int(mirror[0]), int(mirror[1])
        if not mutable[mirror_row, mirror_col]:
            continue
        pixels = tuple(sorted({(int(row), int(col)), (mirror_row, mirror_col)}))
        if pixels in seen:
            continue
        seen.add(pixels)
        groups.append([('m9', r, c) for r, c in pixels])
    if not groups:
        raise RuntimeError('No y-axis symmetric mutable M9 flip groups are available')
    return groups


def sample_y_symmetric_m9_flips(maskset, rng, target_pixel_count):
    groups = symmetric_m9_flip_groups(maskset)
    order = rng.permutation(len(groups))
    flips = []
    for idx in order:
        flips.extend(groups[int(idx)])
        if len(flips) >= int(target_pixel_count):
            break
    return flips


def legality_reasons(maskset, cfg):
    return check_legality(maskset, cfg).reasons


def propose_legal_y_symmetric_m9_candidate(current_maskset, cfg, rng, target_pixel_count, max_attempts=100):
    last_reasons = []
    for attempt in range(1, max_attempts + 1):
        flips = sample_y_symmetric_m9_flips(current_maskset, rng, target_pixel_count)
        candidate = apply_flips(current_maskset, flips)
        assert_y_axis_symmetric(candidate, 'm9')
        reasons = legality_reasons(candidate, cfg)
        if not reasons:
            return candidate, flips, attempt
        last_reasons = reasons
    raise RuntimeError(f'Could not find a legal symmetric M9 proposal after {max_attempts} attempts; last reasons={last_reasons}')


rng = np.random.default_rng(cfg.dbs.random_seed)
current = rasterize_config(cfg)
assert_y_axis_symmetric(current, 'm9')
base_reasons = legality_reasons(current, cfg)
if base_reasons:
    raise RuntimeError(f'Base layout is illegal before DBS: {base_reasons}')

runner = None  # select from cfg; fake by default in this public notebook
best_fom = -1e30
events = []

base_event = evaluate_masks(cfg, current.copy(), run_dir, 0, runner=runner, incumbent_fom=best_fom)
base_event['y_axis_symmetric_flips'] = True
base_event['proposal_attempts'] = 0
base_event['legality_reasons'] = []
events.append(base_event)
if base_event.get('objective_valid'):
    best_fom = float(base_event['fom'])

for eval_index in range(1, int(cfg.dbs.max_evaluations)):
    candidate, flips, proposal_attempts = propose_legal_y_symmetric_m9_candidate(
        current,
        cfg,
        rng,
        target_pixel_count=cfg.dbs.metal_flip_count_values[0],
    )
    event = evaluate_masks(cfg, candidate, run_dir, eval_index, runner=runner, incumbent_fom=best_fom)
    event['y_axis_symmetric_flips'] = True
    event['m9_only_flips'] = flips
    event['proposal_attempts'] = proposal_attempts
    event['legality_reasons'] = []
    events.append(event)
    if event.get('objective_valid') and event.get('accepted'):
        current = candidate
        best_fom = float(event['fom'])

trial_rows = []
for event in events:
    metrics = event.get('metrics') or {}
    trial_rows.append(
        {
            'eval': event.get('eval_index'),
            'accepted': event.get('accepted'),
            'fom_min_q': event.get('fom'),
            'q_primary': metrics.get('q_primary'),
            'q_secondary': metrics.get('q_secondary'),
            'actual_freq_ghz': metrics.get('actual_freq_ghz'),
            'proposal_attempts': event.get('proposal_attempts'),
            'y_axis_symmetric': event.get('y_axis_symmetric_flips', event.get('eval_index') == 0),
        }
    )
trial_table = pd.DataFrame(trial_rows)
display(trial_table)

trial_preview = out_dir / 'q_symmetric_dbs_latest_layout.png'
write_layout_preview(current, trial_preview, cfg, annotate_geometry=False, show_legend=False, show_title=True)
display(Image(filename=str(trial_preview)))


## 8. Make A Real-EMX Local Config

Keep real EMX paths, process keys, license variables, and host-specific setup in ignored local files. The example below writes a local YAML template under `local/`. Edit it on the EMX host before setting `backend: real` for production work.


In [ ]:
real_template = yaml.safe_load(config_path.read_text())
real_template['emx'].update(
    {
        'backend': 'real',
        'executable': 'emx',
        'proc_file': '/absolute/path/to/process_file.proc',
        'env_script': '/absolute/path/to/setup_emx_env.sh',
        'key': None,
        'extra_args': [],
        'timeout_s': 900,
        'retries': 1,
    }
)
real_template_path = out_dir / 'public_dual_core_vco_tank_ring33.real.local.yaml'
real_template_path.write_text(yaml.safe_dump(real_template, sort_keys=False), encoding='utf-8')

print('Wrote local real-EMX template:', real_template_path)
print(yaml.safe_dump({'emx': real_template['emx'], 'ports': real_template['ports']}, sort_keys=False))


A source-safe environment script should set modules, tool paths, and license variables, then return to the caller. Avoid `exit`, `cd`, or commands that deactivate your Python virtual environment.


In [ ]:
setup_script_template = '''#!/usr/bin/env bash
# Source-safe EMX environment setup. Keep this file local and untracked.
# module load <cadence-or-emx-module>
# source <site-cadence-setup.sh>
# export CDS_LIC_FILE=<license-server>
# export LM_LICENSE_FILE=<license-server>
# command -v emx >/dev/null
'''
print(setup_script_template)


## 9. Run `emx-dbs`

The fake backend commands below are safe to run anywhere. For a real EMX host, use the local real-EMX config template from the previous section after replacing the placeholder paths.


In [ ]:
fake_commands = f'''
cd "{repo_root}"
source .venv/bin/activate
python -m pip install -e ".[dev]"
export PYTHONPATH="{out_dir}:$PYTHONPATH"

emx-dbs validate-env "{config_path}"
emx-dbs inspect-gds "{config_path}"
emx-dbs preview-input "{config_path}" --output "{out_dir / 'configured_input_preview.png'}"
emx-dbs rasterize "{config_path}"
emx-dbs eval-one "{config_path}"
emx-dbs report "{run_dir}" --summary-only
'''
print(fake_commands)


Run one fake evaluation from Python to confirm the generated GDS, config, EMX runner, Touchstone parser, and objective all agree. This writes artifacts under `local/`.


In [ ]:
fake_run_dir = eval_one(config_path)
summary = generate_report(fake_run_dir, summary_only=True)
print('Fake run dir:', fake_run_dir)
print(json.dumps(summary, indent=2, sort_keys=True, default=str))

run_script = fake_run_dir / 'evaluations' / 'eval_0000' / 'emx' / 'run_emx.sh'
metrics = fake_run_dir / 'evaluations' / 'eval_0000' / 'results' / 'metrics.json'
print('Generated run script:', run_script)
print(run_script.read_text())
print('Metrics:', metrics)
print(metrics.read_text())


## 10. Files Written By This Notebook

The generated public notebook is tracked, but the GDS, YAML, previews, and run outputs below live under ignored `local/` paths.


In [ ]:
for path in sorted(out_dir.rglob('*')):
    if path.is_file() and '__pycache__' not in path.parts:
        print(path.relative_to(repo_root))
